In [1]:
%pip install -q transformers datasets accelerate evaluate scikit-learn optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 41.6 MB/s eta 0:00:00


In [2]:
pip install optuna

In [3]:
# Imports and seed

import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import optuna

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [22]:
# Upload cleaned dataset

uploaded = files.upload()

Saving df_filtered_cleaned.csv to df_filtered_cleaned.csv


In [23]:
# Load cleaned dataset

df_filtered = pd.read_csv("df_filtered_cleaned.csv")

print("Original shape:", df_filtered.shape)
df_filtered.head()

Original shape: (32030, 3)


,author_ID,post_cleaned,political_leaning
0,t2_7ramzeng,you can buy the show and stream it through the...,right
1,t2_7ramzeng,me want to play q bert holy shit based alex jo...,right
2,t2_7ramzeng,shouldn t rely on any external services or per...,right
3,t2_7ramzeng,pr to a specific person usually that just mean...,right
4,t2_7ramzeng,this article s intention is clear that they wa...,right


In [7]:
# Basic data checks

required_columns = ["author_ID", "post_cleaned", "political_leaning"]

for col in required_columns:
    print(col, "missing:", df_filtered[col].isna().sum())

df_filtered = df_filtered.dropna(subset=required_columns).copy()

df_filtered["author_ID"] = df_filtered["author_ID"].astype(str)
df_filtered["post_cleaned"] = df_filtered["post_cleaned"].astype(str)
df_filtered["political_leaning"] = df_filtered["political_leaning"].astype(str)

print("After dropping missing:", df_filtered.shape)

print("\nClass counts:")
print(df_filtered["political_leaning"].value_counts())

print("\nClass distribution:")
print(df_filtered["political_leaning"].value_counts(normalize=True))

author_ID missing: 0
post_cleaned missing: 0
political_leaning missing: 0
After dropping missing: (32030, 3)

Class counts:
political_leaning
right    17454
left     14576
Name: count, dtype: int64

Class distribution:
political_leaning
right    0.544927
left     0.455073
Name: proportion, dtype: float64


In [8]:
# Author level train test split

unique_authors = df_filtered["author_ID"].unique()

train_authors, test_authors = train_test_split(
    unique_authors,
    test_size=0.20,
    random_state=SEED
)

train_df = df_filtered[df_filtered["author_ID"].isin(train_authors)].copy()
test_df = df_filtered[df_filtered["author_ID"].isin(test_authors)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

overlap_authors = set(train_df["author_ID"]).intersection(set(test_df["author_ID"]))
print("Overlapping authors:", len(overlap_authors))

print("\nTrain class distribution:")
print(train_df["political_leaning"].value_counts(normalize=True))

print("\nTest class distribution:")
print(test_df["political_leaning"].value_counts(normalize=True))

Train shape: (26606, 3)
Test shape: (5424, 3)
Overlapping authors: 0

Train class distribution:
political_leaning
right    0.548185
left     0.451815
Name: proportion, dtype: float64

Test class distribution:
political_leaning
right    0.528945
left     0.471055
Name: proportion, dtype: float64


In [9]:
# Internal author level train validation split

train_unique_authors = train_df["author_ID"].unique()

mb_train_authors, mb_val_authors = train_test_split(
    train_unique_authors,
    test_size=0.20,
    random_state=SEED
)

mb_train_df = train_df[train_df["author_ID"].isin(mb_train_authors)].copy()
mb_val_df = train_df[train_df["author_ID"].isin(mb_val_authors)].copy()

print("ModernBERT train shape:", mb_train_df.shape)
print("ModernBERT validation shape:", mb_val_df.shape)

overlap_mb_authors = set(mb_train_df["author_ID"]).intersection(set(mb_val_df["author_ID"]))
print("Overlapping authors between train and validation:", len(overlap_mb_authors))

print("\nModernBERT train distribution:")
print(mb_train_df["political_leaning"].value_counts(normalize=True))

print("\nModernBERT validation distribution:")
print(mb_val_df["political_leaning"].value_counts(normalize=True))

ModernBERT train shape: (22137, 3)
ModernBERT validation shape: (4469, 3)
Overlapping authors between train and validation: 0

ModernBERT train distribution:
political_leaning
right    0.550752
left     0.449248
Name: proportion, dtype: float64

ModernBERT validation distribution:
political_leaning
right    0.535467
left     0.464533
Name: proportion, dtype: float64


In [10]:
# Prepare labels for ModernBERT

label2id = {
    "left": 0,
    "right": 1
}

id2label = {
    0: "left",
    1: "right"
}

mb_train_df["label"] = mb_train_df["political_leaning"].map(label2id)
mb_val_df["label"] = mb_val_df["political_leaning"].map(label2id)
test_df["label"] = test_df["political_leaning"].map(label2id)

print("ModernBERT train labels:")
print(mb_train_df["label"].value_counts())

print("\nModernBERT validation labels:")
print(mb_val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

print("\nAny unmapped labels?")
print("Train:", mb_train_df["label"].isna().sum())
print("Validation:", mb_val_df["label"].isna().sum())
print("Test:", test_df["label"].isna().sum())

ModernBERT train labels:
label
1    12192
0     9945
Name: count, dtype: int64

ModernBERT validation labels:
label
1    2393
0    2076
Name: count, dtype: int64

Test labels:
label
1    2869
0    2555
Name: count, dtype: int64

Any unmapped labels?
Train: 0
Validation: 0
Test: 0


In [11]:
# Load ModernBERT tokenizer

model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [12]:
# Metrics for ModernBERT

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1
    }

In [13]:
# Train validation test performance table

def evaluate_train_validation_test(
    trainer,
    train_dataset,
    validation_dataset,
    test_dataset,
    model_name_for_table,
    extra_info=None
):
    train_results = trainer.evaluate(train_dataset)
    validation_results = trainer.evaluate(validation_dataset)
    test_results = trainer.evaluate(test_dataset)

    row = {
        "model": model_name_for_table,
        "train_accuracy": train_results["eval_accuracy"],
        "validation_accuracy": validation_results["eval_accuracy"],
        "test_accuracy": test_results["eval_accuracy"],
        "train_macro_f1": train_results["eval_macro_f1"],
        "validation_macro_f1": validation_results["eval_macro_f1"],
        "test_macro_f1": test_results["eval_macro_f1"],
        "train_validation_accuracy_gap": (
            train_results["eval_accuracy"] - validation_results["eval_accuracy"]
        ),
        "train_test_accuracy_gap": (
            train_results["eval_accuracy"] - test_results["eval_accuracy"]
        ),
        "train_validation_macro_f1_gap": (
            train_results["eval_macro_f1"] - validation_results["eval_macro_f1"]
        ),
        "train_test_macro_f1_gap": (
            train_results["eval_macro_f1"] - test_results["eval_macro_f1"]
        )
    }

    if extra_info is not None:
        row.update(extra_info)

    return pd.DataFrame([row])

In [14]:
# GPU memory helper

def clear_gpu_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [15]:
# Dataset builder for ModernBERT Optuna

def build_modernbert_dataset(max_length):
    def tokenize_function_optuna(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length
        )

    train_dataset = Dataset.from_pandas(
        mb_train_df[["post_cleaned", "label"]]
        .rename(columns={"post_cleaned": "text", "label": "labels"}),
        preserve_index=False
    )

    validation_dataset = Dataset.from_pandas(
        mb_val_df[["post_cleaned", "label"]]
        .rename(columns={"post_cleaned": "text", "label": "labels"}),
        preserve_index=False
    )

    train_dataset = train_dataset.map(
        tokenize_function_optuna,
        batched=True,
        remove_columns=["text"]
    )

    validation_dataset = validation_dataset.map(
        tokenize_function_optuna,
        batched=True,
        remove_columns=["text"]
    )

    train_dataset.set_format("torch")
    validation_dataset.set_format("torch")

    data_collator_trial = DataCollatorWithPadding(tokenizer=tokenizer)

    return train_dataset, validation_dataset, data_collator_trial

In [16]:
# Define ModernBERT Optuna objective

def modernbert_objective(trial):
    clear_gpu_memory()

    max_length = trial.suggest_categorical("max_length", [512, 1024])
    learning_rate = trial.suggest_categorical("learning_rate", [1e-5, 2e-5, 3e-5, 5e-5])
    weight_decay = trial.suggest_categorical("weight_decay", [0.01, 0.05, 0.1])
    warmup_ratio = trial.suggest_categorical("warmup_ratio", [0.0, 0.1])

    if max_length == 512:
        batch_size = 8
        gradient_accumulation_steps = 4
    else:
        batch_size = 4
        gradient_accumulation_steps = 8

    train_dataset, validation_dataset, data_collator_trial = build_modernbert_dataset(max_length)

    model_trial = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    training_args_trial = TrainingArguments(
        output_dir=f"modernbert_optuna_trial_{trial.number}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=1,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type="linear",
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_strategy="steps",
        logging_steps=100,
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    trainer_trial = Trainer(
        model=model_trial,
        args=training_args_trial,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=data_collator_trial,
        compute_metrics=compute_metrics
    )

    trainer_trial.train()

    train_results = trainer_trial.evaluate(train_dataset)
    validation_results = trainer_trial.evaluate(validation_dataset)

    train_macro_f1 = train_results["eval_macro_f1"]
    validation_macro_f1 = validation_results["eval_macro_f1"]
    macro_f1_gap = train_macro_f1 - validation_macro_f1

    train_accuracy = train_results["eval_accuracy"]
    validation_accuracy = validation_results["eval_accuracy"]
    accuracy_gap = train_accuracy - validation_accuracy

    trial.set_user_attr("train_accuracy", train_accuracy)
    trial.set_user_attr("validation_accuracy", validation_accuracy)
    trial.set_user_attr("train_macro_f1", train_macro_f1)
    trial.set_user_attr("validation_macro_f1", validation_macro_f1)
    trial.set_user_attr("accuracy_gap", accuracy_gap)
    trial.set_user_attr("macro_f1_gap", macro_f1_gap)
    trial.set_user_attr("batch_size", batch_size)
    trial.set_user_attr("gradient_accumulation_steps", gradient_accumulation_steps)

    del trainer_trial
    del model_trial
    clear_gpu_memory()

    return validation_macro_f1

In [18]:
# Run ModernBERT Optuna

modernbert_study = optuna.create_study(
    study_name="modernbert_optuna_study",
    direction="maximize",
    storage="sqlite:///modernbert_optuna_study.db",
    load_if_exists=True
)

modernbert_study.optimize(
    modernbert_objective,
    n_trials=8
)

print("Best validation Macro F1:", modernbert_study.best_value)
print("Best parameters:")
print(modernbert_study.best_params)

print("\nBest trial diagnostics:")
print("Train Accuracy:", modernbert_study.best_trial.user_attrs["train_accuracy"])
print("Validation Accuracy:", modernbert_study.best_trial.user_attrs["validation_accuracy"])
print("Train Macro F1:", modernbert_study.best_trial.user_attrs["train_macro_f1"])
print("Validation Macro F1:", modernbert_study.best_trial.user_attrs["validation_macro_f1"])
print("Accuracy Gap:", modernbert_study.best_trial.user_attrs["accuracy_gap"])
print("Macro F1 Gap:", modernbert_study.best_trial.user_attrs["macro_f1_gap"])

[I 2026-05-20 16:48:47,693] A new study created in RDB with name: modernbert_optuna_study


GPU memory cleared.


Map:   0%|          | 0/22137 [00:00<?, ? examples/s]

Map:   0%|          | 0/4469 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,2.316405,0.668307,0.609085,0.606473,0.605720,0.605864


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-05-20 18:02:52,655] Trial 7 finished with value: 0.605863824285688 and parameters: {'max_length': 512, 'learning_rate': 1e-05, 'weight_decay': 0.05, 'warmup_ratio': 0.0}. Best is trial 2 with value: 0.6513643073831625.


GPU memory cleared.
Best validation Macro F1: 0.6513643073831625
Best parameters:
{'max_length': 1024, 'learning_rate': 3e-05, 'weight_decay': 0.1, 'warmup_ratio': 0.0}

Best trial diagnostics:
Train Accuracy: 0.8185390974386774
Validation Accuracy: 0.6558514208995301
Train Macro F1: 0.8150512208823613
Validation Macro F1: 0.6513643073831625
Accuracy Gap: 0.16268767653914729
Macro F1 Gap: 0.16368691349919873


In [19]:
# Save ModernBERT Optuna trial results

modernbert_optuna_results = []

for trial in modernbert_study.trials:
    if trial.value is not None:
        row = {
            "trial": trial.number,
            "objective_value": trial.value,
            **trial.params,
            **trial.user_attrs
        }
        modernbert_optuna_results.append(row)

modernbert_optuna_results_df = pd.DataFrame(modernbert_optuna_results)

modernbert_optuna_results_df.to_csv(
    "modernbert_optuna_trial_results.csv",
    index=False
)

modernbert_optuna_results_df.sort_values(
    by="validation_macro_f1",
    ascending=False
)

,trial,objective_value,max_length,learning_rate,weight_decay,warmup_ratio,accuracy_gap,batch_size,gradient_accumulation_steps,macro_f1_gap,train_accuracy,train_macro_f1,validation_accuracy,validation_macro_f1
2,2,0.651364,1024,0.00003,0.10,0.0,0.162688,4,8,0.163687,0.818539,0.815051,0.655851,0.651364
1,1,0.647235,1024,0.00003,0.10,0.0,0.159896,4,8,0.161262,0.811944,0.808497,0.652047,0.647235
0,0,0.637144,512,0.00005,0.05,0.1,0.161409,8,4,0.161148,0.801373,0.798292,0.639964,0.637144
4,4,0.625982,512,0.00005,0.01,0.0,0.178482,8,4,0.179282,0.808601,0.805264,0.630119,0.625982
5,5,0.625147,1024,0.00002,0.01,0.1,0.147172,4,8,0.150134,0.781542,0.775281,0.634370,0.625147
7,7,0.605864,512,0.00001,0.05,0.0,0.102891,8,4,0.101574,0.711975,0.707438,0.609085,0.605864
3,3,0.603487,512,0.00001,0.01,0.1,0.102474,8,4,0.101758,0.710440,0.705244,0.607966,0.603487
6,6,0.588843,512,0.00001,0.10,0.0,0.109865,8,4,0.110895,0.706419,0.699739,0.596554,0.588843


In [20]:
# Build datasets for best ModernBERT configuration

best_modernbert_params = modernbert_study.best_params

BEST_MAX_LENGTH = best_modernbert_params["max_length"]

print("Best ModernBERT parameters:")
print(best_modernbert_params)

def tokenize_function_best(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=BEST_MAX_LENGTH
    )

best_train_dataset = Dataset.from_pandas(
    mb_train_df[["post_cleaned", "label"]]
    .rename(columns={"post_cleaned": "text", "label": "labels"}),
    preserve_index=False
)

best_validation_dataset = Dataset.from_pandas(
    mb_val_df[["post_cleaned", "label"]]
    .rename(columns={"post_cleaned": "text", "label": "labels"}),
    preserve_index=False
)

best_test_dataset = Dataset.from_pandas(
    test_df[["post_cleaned", "label"]]
    .rename(columns={"post_cleaned": "text", "label": "labels"}),
    preserve_index=False
)

best_train_dataset = best_train_dataset.map(
    tokenize_function_best,
    batched=True,
    remove_columns=["text"]
)

best_validation_dataset = best_validation_dataset.map(
    tokenize_function_best,
    batched=True,
    remove_columns=["text"]
)

best_test_dataset = best_test_dataset.map(
    tokenize_function_best,
    batched=True,
    remove_columns=["text"]
)

best_train_dataset.set_format("torch")
best_validation_dataset.set_format("torch")
best_test_dataset.set_format("torch")

best_data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(best_train_dataset)
print(best_validation_dataset)
print(best_test_dataset)

Best ModernBERT parameters:
{'max_length': 1024, 'learning_rate': 3e-05, 'weight_decay': 0.1, 'warmup_ratio': 0.0}


Map:   0%|          | 0/22137 [00:00<?, ? examples/s]

Map:   0%|          | 0/4469 [00:00<?, ? examples/s]

Map:   0%|          | 0/5424 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 22137
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 4469
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 5424
})


In [21]:
# Train final candidate ModernBERT using best Optuna parameters

clear_gpu_memory()

model_modernbert_final_candidate = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

if BEST_MAX_LENGTH == 512:
    best_batch_size = 8
    best_gradient_accumulation_steps = 4
else:
    best_batch_size = 4
    best_gradient_accumulation_steps = 8

training_args_modernbert_final_candidate = TrainingArguments(
    output_dir="modernbert_final_candidate",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_modernbert_params["learning_rate"],
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    gradient_accumulation_steps=best_gradient_accumulation_steps,
    num_train_epochs=1,
    weight_decay=best_modernbert_params["weight_decay"],
    warmup_ratio=best_modernbert_params["warmup_ratio"],
    lr_scheduler_type="linear",
    optim="adamw_torch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer_modernbert_final_candidate = Trainer(
    model=model_modernbert_final_candidate,
    args=training_args_modernbert_final_candidate,
    train_dataset=best_train_dataset,
    eval_dataset=best_validation_dataset,
    data_collator=best_data_collator,
    compute_metrics=compute_metrics
)

trainer_modernbert_final_candidate.train()

GPU memory cleared.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,3.710694,0.671776,0.648915,0.647282,0.643295,0.643355


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=692, training_loss=4.349785336180229, metrics={'train_runtime': 523.4023, 'train_samples_per_second': 42.294, 'train_steps_per_second': 1.322, 'total_flos': 1.5086712874512384e+16, 'train_loss': 4.349785336180229, 'epoch': 1.0})

In [22]:
# Final candidate ModernBERT train validation test performance

modernbert_final_candidate_results = evaluate_train_validation_test(
    trainer=trainer_modernbert_final_candidate,
    train_dataset=best_train_dataset,
    validation_dataset=best_validation_dataset,
    test_dataset=best_test_dataset,
    model_name_for_table="ModernBERT Optuna final candidate",
    extra_info={
        "max_length": best_modernbert_params["max_length"],
        "learning_rate": best_modernbert_params["learning_rate"],
        "epochs": 1,
        "weight_decay": best_modernbert_params["weight_decay"],
        "warmup_ratio": best_modernbert_params["warmup_ratio"],
        "batch_size": best_batch_size,
        "gradient_accumulation_steps": best_gradient_accumulation_steps,
        "effective_batch_size": best_batch_size * best_gradient_accumulation_steps
    }
)

modernbert_final_candidate_results.to_csv(
    "modernbert_final_candidate_train_validation_test_results.csv",
    index=False
)

modernbert_final_candidate_results

,model,train_accuracy,validation_accuracy,test_accuracy,train_macro_f1,validation_macro_f1,test_macro_f1,train_validation_accuracy_gap,train_test_accuracy_gap,train_validation_macro_f1_gap,train_test_macro_f1_gap,max_length,learning_rate,epochs,weight_decay,warmup_ratio,batch_size,gradient_accumulation_steps,effective_batch_size
0,ModernBERT Optuna final candidate,0.822334,0.648915,0.654867,0.819055,0.643355,0.649776,0.173419,0.167466,0.1757,0.169279,1024,0.00003,1,0.1,0.0,4,8,32


In [23]:
# Build full train and test datasets

FINAL_MAX_LENGTH = best_modernbert_params["max_length"]

def tokenize_final(batch):
    return tokenizer(batch["text"], truncation=True, max_length=FINAL_MAX_LENGTH)

final_train_df = train_df.copy()
final_train_df["label"] = final_train_df["political_leaning"].map(label2id)

final_train_dataset = Dataset.from_pandas(
    final_train_df[["post_cleaned", "label"]]
    .rename(columns={"post_cleaned": "text", "label": "labels"}),
    preserve_index=False
).map(tokenize_final, batched=True, remove_columns=["text"])

final_test_dataset = Dataset.from_pandas(
    test_df[["post_cleaned", "label"]]
    .rename(columns={"post_cleaned": "text", "label": "labels"}),
    preserve_index=False
).map(tokenize_final, batched=True, remove_columns=["text"])

final_train_dataset.set_format("torch")
final_test_dataset.set_format("torch")

print("Final train:", len(final_train_dataset))
print("Final test:", len(final_test_dataset))

Map:   0%|          | 0/26606 [00:00<?, ? examples/s]

Map:   0%|          | 0/5424 [00:00<?, ? examples/s]

Final train: 26606
Final test: 5424


In [25]:
# Train final ModernBERT on full train set

clear_gpu_memory()

model_final = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args_final = TrainingArguments(
    output_dir="modernbert_final",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=best_modernbert_params["learning_rate"],
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    gradient_accumulation_steps=best_gradient_accumulation_steps,
    num_train_epochs=2,
    weight_decay=best_modernbert_params["weight_decay"],
    warmup_steps=200,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=True,
    fp16=False,
    report_to="none"
)

trainer_final = Trainer(
    model=model_final,
    args=training_args_final,
    train_dataset=final_train_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer_final.train()

# Test evaluation
test_results = trainer_final.evaluate(final_test_dataset)
train_results = trainer_final.evaluate(final_train_dataset)

print("\nTrain macro F1:", round(train_results["eval_macro_f1"], 4))
print("Test macro F1:", round(test_results["eval_macro_f1"], 4))
print("Gap:", round(train_results["eval_macro_f1"] - test_results["eval_macro_f1"], 4))

GPU memory cleared.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,4.982294
1000,3.612236
1500,2.782178


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Train macro F1: 0.914
Test macro F1: 0.6624
Gap: 0.2516


In [31]:
# Train final ModernBERT on full train set

clear_gpu_memory()

# Author cap
final_train_df = train_df.copy()
final_train_df = (
    final_train_df
    .groupby("author_ID", group_keys=False)
    .head(30)
    .reset_index(drop=True)
)
final_train_df["label"] = final_train_df["political_leaning"].map(label2id)
print("Capped final train:", final_train_df.shape)

# Retokenize
final_train_dataset = Dataset.from_pandas(
    final_train_df[["post_cleaned", "label"]]
    .rename(columns={"post_cleaned": "text", "label": "labels"}),
    preserve_index=False
).map(tokenize_final, batched=True, remove_columns=["text"])
final_train_dataset.set_format("torch")

model_final = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args_final = TrainingArguments(
    output_dir="modernbert_final",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    gradient_accumulation_steps=best_gradient_accumulation_steps,
    num_train_epochs=2,
    weight_decay=0.2,
    warmup_steps=300,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=True,
    fp16=False,
    report_to="none"
)

trainer_final = Trainer(
    model=model_final,
    args=training_args_final,
    train_dataset=final_train_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer_final.train()

test_results = trainer_final.evaluate(final_test_dataset)
train_results = trainer_final.evaluate(final_train_dataset)

print("\nTrain macro F1:", round(train_results["eval_macro_f1"], 4))
print("Test macro F1:", round(test_results["eval_macro_f1"], 4))
print("Gap:", round(train_results["eval_macro_f1"] - test_results["eval_macro_f1"], 4))

GPU memory cleared.
Capped final train: (14679, 4)


Map:   0%|          | 0/14679 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,5.205146


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Train macro F1: 0.7561
Test macro F1: 0.662
Gap: 0.0941


In [32]:
# Applying Lora

In [33]:
!pip install peft -q
from peft import LoraConfig, get_peft_model, TaskType

PEFT ready.


In [36]:
from peft import LoraConfig, get_peft_model, TaskType

clear_gpu_memory()

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["Wqkv", "Wi_0", "Wi_1", "Wo"]
)

model_lora = get_peft_model(base_model, lora_config)
model_lora.print_trainable_parameters()

GPU memory cleared.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 2,299,394 || all params: 151,905,796 || trainable%: 1.5137


In [37]:
training_args_lora = TrainingArguments(
    output_dir="modernbert_lora",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=True,
    fp16=False,
    report_to="none"
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=final_train_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer_lora.train()

test_results_lora = trainer_lora.evaluate(final_test_dataset)
train_results_lora = trainer_lora.evaluate(final_train_dataset)

print("\nTrain macro F1:", round(train_results_lora["eval_macro_f1"], 4))
print("Test macro F1:", round(test_results_lora["eval_macro_f1"], 4))
print("Gap:", round(train_results_lora["eval_macro_f1"] - test_results_lora["eval_macro_f1"], 4))

Step,Training Loss
500,2.604848
1000,2.292258



Train macro F1: 0.7409
Test macro F1: 0.6617
Gap: 0.0792


In [38]:
def lora_objective(trial):
    clear_gpu_memory()

    r = trial.suggest_categorical("r", [8, 16, 32])
    lora_alpha = trial.suggest_categorical("lora_alpha", [16, 32, 64])
    lora_dropout = trial.suggest_float("lora_dropout", 0.05, 0.2)
    learning_rate = trial.suggest_categorical("learning_rate", [1e-4, 2e-4, 3e-4, 5e-4])
    epochs = trial.suggest_categorical("epochs", [2, 3, 5])

    base = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=id2label, label2id=label2id
    )

    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
        target_modules=["Wqkv", "Wi_0", "Wi_1", "Wo"]
    )

    model_trial = get_peft_model(base, config)

    args = TrainingArguments(
        output_dir=f"lora_trial_{trial.number}",
        eval_strategy="no",
        save_strategy="no",
        learning_rate=learning_rate,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=200,
        seed=SEED,
        bf16=True,
        fp16=False,
        report_to="none"
    )

    trainer_trial = Trainer(
        model=model_trial,
        args=args,
        train_dataset=final_train_dataset,
        eval_dataset=best_validation_dataset,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer_trial.train()
    results = trainer_trial.evaluate()

    del model_trial, trainer_trial
    clear_gpu_memory()

    return results["eval_macro_f1"]

lora_study = optuna.create_study(direction="maximize")
lora_study.optimize(lora_objective, n_trials=8)

print("Best val macro F1:", round(lora_study.best_value, 4))
print("Best params:", lora_study.best_params)

[I 2026-05-20 20:15:32,457] A new study created in memory with name: no-name-db1355d0-3300-44e4-8334-d46ec07f170f


GPU memory cleared.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
W0520 20:15:34.651000 640 torch/_dynamo/convert_frame.py:1676] [1/8] torch._dynamo hit config.recompile_limit (8)
W0520 20:15:34.651000 640 torch/_dynamo/convert_frame.py:1676] [1/8]    function: 'compiled_mlp' (/usr/local/lib/python3.12/dist-packages/transformers/models/modernbert/modeling_modernbert.py:580)
W0520 20:15:34.651000 640 torch/_dynamo/convert_frame.py:1676] [1/8]    last reason: 1/7: GLOBAL_STATE changed: grad_mode 
W0520 20:15:34.651000 640 torch/_dynamo/convert_frame.py:1676] [1/

Step,Training Loss
500,2.601305
1000,2.324861


...

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,2.660655
1000,2.410155
1500,2.291520
2000,2.177743


[I 2026-05-20 22:45:16,469] Trial 7 finished with value: 0.7368011690048193 and parameters: {'r': 32, 'lora_alpha': 32, 'lora_dropout': 0.18292886926269625, 'learning_rate': 0.0001, 'epochs': 5}. Best is trial 1 with value: 0.8744086326935164.


GPU memory cleared.
Best val macro F1: 0.8744
Best params: {'r': 32, 'lora_alpha': 64, 'lora_dropout': 0.17661186749757463, 'learning_rate': 0.0005, 'epochs': 5}


In [39]:
clear_gpu_memory()

base_final = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id
)

best_lora_params = lora_study.best_params

lora_config_final = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=best_lora_params["r"],
    lora_alpha=best_lora_params["lora_alpha"],
    lora_dropout=best_lora_params["lora_dropout"],
    target_modules=["Wqkv", "Wi_0", "Wi_1", "Wo"]
)

model_lora_final = get_peft_model(base_final, lora_config_final)

training_args_lora_final = TrainingArguments(
    output_dir="lora_final",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=best_lora_params["learning_rate"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=best_lora_params["epochs"],
    weight_decay=0.01,
    warmup_steps=200,
    seed=SEED,
    bf16=True,
    fp16=False,
    report_to="none"
)

trainer_lora_final = Trainer(
    model=model_lora_final,
    args=training_args_lora_final,
    train_dataset=final_train_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer_lora_final.train()

test_results_lora = trainer_lora_final.evaluate(final_test_dataset)
train_results_lora = trainer_lora_final.evaluate(final_train_dataset)

print("\nTrain macro F1:", round(train_results_lora["eval_macro_f1"], 4))
print("Test macro F1:", round(test_results_lora["eval_macro_f1"], 4))
print("Gap:", round(train_results_lora["eval_macro_f1"] - test_results_lora["eval_macro_f1"], 4))

GPU memory cleared.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,2.627350
1000,2.238759
1500,1.837948
2000,1.290141



Train macro F1: 0.9605
Test macro F1: 0.6684
Gap: 0.2921


In [41]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("modernbert_final", "/content/drive/MyDrive/modernbert_final")

Mounted at /content/drive
Saved to Drive.


In [42]:
import os
for d in os.listdir("."):
    if os.path.isdir(d) and "modernbert" in d.lower():
        print(d)

modernbert_optuna_trial_2
modernbert_optuna_trial_1
modernbert_final
modernbert_optuna_trial_3
modernbert_optuna_trial_4
modernbert_optuna_trial_5
modernbert_optuna_trial_7
modernbert_optuna_trial_0
modernbert_optuna_trial_6
modernbert_final_candidate
modernbert_lora


In [43]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("modernbert_lora", "/content/drive/MyDrive/modernbert_lora")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive.


In [44]:
# Base model
trainer_lora.model.save_pretrained("/content/drive/MyDrive/modernbert_lora_full")
tokenizer.save_pretrained("/content/drive/MyDrive/modernbert_lora_full")

Full model saved.


In [45]:
from peft import PeftModel
merged_model = trainer_lora.model.merge_and_unload()
merged_model.save_pretrained("/content/drive/MyDrive/modernbert_lora_merged")
tokenizer.save_pretrained("/content/drive/MyDrive/modernbert_lora_merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved.


# SHAP/LIME Ablation

In [1]:
!pip install shap lime -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 24.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [11]:
# Importing libraries
import shap
from lime.lime_text import LimeTextExplainer
import pandas as pd
import numpy as np
import re
from collections import defaultdict


In [13]:
# Load lexicons
from google.colab import files
uploaded = files.upload()

Saving eu-political-barometer_keywords_2024_v01.csv to eu-political-barometer_keywords_2024_v01.csv
Saving lexicon.csv to lexicon.csv


In [14]:
# Load lexicons
eu_keywords = pd.read_csv("eu-political-barometer_keywords_2024_v01.csv")
eu_keywords["word"] = eu_keywords["word"].astype(str).str.lower().str.strip()

harvard_lexicon = pd.read_csv("lexicon.csv", header=None, names=["word"])
harvard_lexicon["word"] = harvard_lexicon["word"].astype(str).str.lower().str.strip()

# Combine into one set
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

political_lexicon = set(eu_keywords["word"]).union(set(harvard_lexicon["word"]))
political_lexicon = {w for w in political_lexicon if w not in ENGLISH_STOP_WORDS and len(w) > 2}

print("EU lexicon size:", len(eu_keywords))
print("Harvard lexicon size:", len(harvard_lexicon))
print("Combined lexicon size:", len(political_lexicon))

EU lexicon size: 15927046
Harvard lexicon size: 94
Combined lexicon size: 19998


In [15]:
# Load and clean lexicons
eu_keywords = pd.read_csv("eu-political-barometer_keywords_2024_v01.csv")
print("EU columns:", eu_keywords.columns.tolist())
print(eu_keywords.head())

harvard_lexicon = pd.read_csv("lexicon.csv", header=None, names=["word"])
print("\nHarvard head:")
print(harvard_lexicon.head())

EU columns: ['party_id', 'word', 'week', 'freq']
   party_id      word        week  freq
0      2612    accord  2021-10-10     1
1      2612    afraid  2021-10-10     1
2      2612    anyone  2021-10-10     1
3      2612  anywhere  2021-10-10     1
4      2612       ban  2021-10-10     1

Harvard head:
             word
0  ADMINISTRATION
1       ADVERSARY
2        ALLIANCE
3          ALLIED
4            ALLY


In [16]:
# Clean EU lexicon
eu_words = set(eu_keywords["word"].astype(str).str.lower().str.strip())

# Clean Harvard lexicon
harvard_words = set(harvard_lexicon["word"].astype(str).str.lower().str.strip())

# Combine
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

political_lexicon = eu_words.union(harvard_words)
political_lexicon = {w for w in political_lexicon if w not in ENGLISH_STOP_WORDS and len(w) > 2}

print("EU lexicon size:", len(eu_words))
print("Harvard lexicon size:", len(harvard_words))
print("Combined lexicon size:", len(political_lexicon))

EU lexicon size: 20317
Harvard lexicon size: 89
Combined lexicon size: 19998


In [24]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split

df = pd.read_csv("df_filtered_cleaned.csv")

unique_authors = df["author_ID"].unique()
train_authors, test_authors = train_test_split(unique_authors, test_size=0.20, random_state=42)

test_df = df[df["author_ID"].isin(test_authors)].copy().reset_index(drop=True)
test_sample = test_df.sample(n=200, random_state=42).reset_index(drop=True)

print("test_sample:", test_sample.shape)

test_sample: (200, 3)


In [25]:
# Prepare test sample for SHAP and LIME

sample_size = 200
test_sample = test_df.sample(n=sample_size, random_state=42).reset_index(drop=True)

print("Test sample shape:", test_sample.shape)
print(test_sample["political_leaning"].value_counts())

Test sample shape: (200, 3)
political_leaning
right    102
left      98
Name: count, dtype: int64


In [27]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_lora_final = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/modernbert_lora_merged"
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/modernbert_lora_merged"
)

id2label = {0: "left", 1: "right"}
label2id = {"left": 0, "right": 1}


Mounted at /content/drive


Loading weights:   0%|          | 0/138 [00:03<?, ?it/s]

Model loaded.


In [28]:
# Prediction function for SHAP and LIME
import torch

def predict_proba(texts):
    inputs = tokenizer(
        list(texts),
        truncation=True,
        max_length=512,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model_lora_final(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

# Test it
sample_text = [test_sample["post_cleaned"].iloc[0]]
result = predict_proba(sample_text)
print("Class order:", list(id2label.values()))
print("Probabilities:", result)

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Class order: ['left', 'right']
Probabilities: [[0.8549725  0.14502747]]


In [29]:
# SHAP

import shap

# Wrap predict function for SHAP
def predict_for_shap(texts):
    return predict_proba(texts)

# Use a small masker for text
masker = shap.maskers.Text(tokenizer=r"\W+")

explainer = shap.Explainer(predict_for_shap, masker, output_names=["left", "right"])

# Explain a small sample
shap_sample = test_sample["post_cleaned"].iloc[:50].tolist()

shap_values = explainer(shap_sample)

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   2%|▏         | 1/50 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   6%|▌         | 3/50 [00:20<03:02,  3.89s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

...

PartitionExplainer explainer:  96%|█████████▌| 48/50 [05:58<00:15,  7.81s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 50/50 [06:13<00:00,  7.60s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 51it [06:20,  7.62s/it]


In [31]:
shap_top_200 = pd.read_csv("/content/drive/MyDrive/modernbert_shap_top_200.csv")
shap_top_200.head(10)

,token,mean_shap_left,mean_shap_right,mean_abs_shap,direction
0,emotions,-0.019924,0.019924,0.019924,right
1,pentup,-0.019924,0.019924,0.019924,right
2,christ,-0.018500,0.018500,0.018979,right
3,bisexual,0.016643,-0.016643,0.016643,left
4,libertarianfreestate,-0.016073,0.016073,0.016073,right
5,opposed,-0.014870,0.014870,0.014870,right
6,firefox,0.012962,-0.012962,0.012962,left
7,konden,0.012813,-0.012813,0.012813,left
8,comprendre,0.012813,-0.012813,0.012813,left
9,manipulating,0.012029,-0.012029,0.012029,left


In [32]:
# LIME setup

from lime.lime_text import LimeTextExplainer

class_names = ["left", "right"]

lime_explainer = LimeTextExplainer(class_names=class_names)

def predict_proba_lime(texts):
    return predict_proba(texts)

In [34]:
import gc
gc.collect()
torch.cuda.empty_cache()

def predict_proba_lime(texts):
    results = []
    for text in texts:
        inputs = tokenizer(
            [text],
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            outputs = model_lora_final(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        results.append(probs[0])
        del inputs, outputs
        torch.cuda.empty_cache()
    return np.array(results)

In [6]:
from collections import defaultdict
import numpy as np

In [36]:
lime_feature_scores = defaultdict(list)
lime_sample_size = 20

np.random.seed(42)
sample_indices = np.random.choice(len(test_sample), size=lime_sample_size, replace=False)

print("Running LIME...")
for i, idx in enumerate(sample_indices):
    text = test_sample["post_cleaned"].iloc[idx]

    exp = lime_explainer.explain_instance(
        text,
        predict_proba_lime,
        num_features=20,
        num_samples=500,
        labels=[0, 1]
    )

    for label_idx in [0, 1]:
        for feature, score in exp.as_list(label=label_idx):
            feature = str(feature).lower().strip()
            lime_feature_scores[feature].append(score if label_idx == 1 else -score)

    if (i + 1) % 5 == 0:
        print(f"{i+1}/{lime_sample_size} done")

Running LIME...
5/20 done
10/20 done
15/20 done
20/20 done


In [37]:
lime_global = pd.DataFrame([
    {
        "feature": feature,
        "mean_abs_lime": np.abs(scores).mean(),
        "mean_lime": np.mean(scores),
        "frequency": len(scores)
    }
    for feature, scores in lime_feature_scores.items()
])

lime_global = lime_global[lime_global["frequency"] >= 2].copy()
lime_global["direction"] = np.where(lime_global["mean_lime"] > 0, "right", "left")
lime_global = lime_global.sort_values("mean_abs_lime", ascending=False).reset_index(drop=True)

lime_top_200 = lime_global.head(200)
lime_top_200.to_csv("/content/drive/MyDrive/modernbert_lime_top_200.csv", index=False)
lime_top_200.head(10)

,feature,mean_abs_lime,mean_lime,frequency,direction
0,government,0.183914,0.183914,8,right
1,libertarian,0.145094,0.145094,4,right
2,guns,0.134727,0.134727,2,right
3,territory,0.113824,0.113824,2,right
4,taxes,0.111792,0.111792,2,right
5,indiana,0.107168,0.107168,2,right
6,are,0.101861,0.101861,6,right
7,capitalist,0.101218,-0.101218,2,left
8,more,0.099142,0.099142,2,right
9,i,0.094998,0.094998,10,right


In [38]:
def feature_overlaps_lexicon(feature, lexicon):
    feature = str(feature).lower().strip()
    if feature in lexicon:
        return True
    tokens = feature.split()
    return any(token in lexicon for token in tokens)

shap_top_200["lexicon_overlap"] = shap_top_200["token"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)

lime_top_200["lexicon_overlap"] = lime_top_200["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)

shap_overlap = shap_top_200[shap_top_200["lexicon_overlap"]].copy()
lime_overlap = lime_top_200[lime_top_200["lexicon_overlap"]].copy()

print("SHAP overlap:", len(shap_overlap), "/", len(shap_top_200))
print("LIME overlap:", len(lime_overlap), "/", len(lime_top_200))

SHAP overlap: 53 / 200
LIME overlap: 75 / 200


/tmp/ipykernel_825/2834098956.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lime_top_200["lexicon_overlap"] = lime_top_200["feature"].apply(


In [39]:
combined_features = set(shap_top_200["token"]).union(set(lime_top_200["feature"]))
combined_df = pd.DataFrame({"feature": sorted(combined_features)})
combined_df["lexicon_overlap"] = combined_df["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)
combined_overlap = combined_df[combined_df["lexicon_overlap"]].copy()

overlap_summary = pd.DataFrame({
    "source": ["SHAP", "LIME", "Combined"],
    "total_features": [len(shap_top_200), len(lime_top_200), len(combined_df)],
    "lexicon_overlap_count": [len(shap_overlap), len(lime_overlap), len(combined_overlap)],
    "overlap_ratio": [
        round(len(shap_overlap)/len(shap_top_200), 3),
        round(len(lime_overlap)/len(lime_top_200), 3),
        round(len(combined_overlap)/len(combined_df), 3)
    ]
})

overlap_summary

,source,total_features,lexicon_overlap_count,overlap_ratio
0,SHAP,200,53,0.265
1,LIME,200,75,0.375
2,Combined,398,127,0.319


In [40]:
overlap_summary.to_csv("/content/drive/MyDrive/modernbert_overlap_summary.csv", index=False)
combined_overlap.to_csv("/content/drive/MyDrive/modernbert_combined_overlap.csv", index=False)

In [43]:
import re

# Ablation terms with combined overlap words
ablation_terms = combined_overlap["feature"].astype(str).str.lower().tolist()
print("Ablation terms:", len(ablation_terms))

def remove_terms(text, terms):
    text = str(text).lower()
    terms_sorted = sorted(terms, key=len, reverse=True)
    for term in terms_sorted:
        pattern = r"\b" + re.escape(term) + r"\b"
        text = re.sub(pattern, " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Create ablated test set
test_df_ablated = test_df.copy()
test_df_ablated["post_cleaned"] = test_df_ablated["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms)
)

Ablation terms: 127


In [45]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

y_true = test_df["political_leaning"].tolist()

# Original model predictions
preds_original = []
batch_size = 8
texts_original = test_df["post_cleaned"].tolist()

for i in range(0, len(texts_original), batch_size):
    batch = texts_original[i:i+batch_size]
    inputs = tokenizer(batch, truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_lora_final(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    preds_original.extend([id2label[p] for p in probs.argmax(axis=1)])

_, _, original_f1, _ = precision_recall_fscore_support(y_true, preds_original, average="macro", zero_division=0)

# Ablated model predictions
preds_ablated = []
texts_ablated = test_df_ablated["post_cleaned"].tolist()

for i in range(0, len(texts_ablated), batch_size):
    batch = texts_ablated[i:i+batch_size]
    inputs = tokenizer(batch, truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_lora_final(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    preds_ablated.extend([id2label[p] for p in probs.argmax(axis=1)])

_, _, ablated_f1, _ = precision_recall_fscore_support(y_true, preds_ablated, average="macro", zero_division=0)

ablation_results = pd.DataFrame({
    "model": ["Original ModernBERT", "Ablated ModernBERT"],
    "macro_f1": [round(original_f1, 4), round(ablated_f1, 4)],
    "f1_change": [0, round(ablated_f1 - original_f1, 4)]
})

ablation_results

,model,macro_f1,f1_change
0,Original ModernBERT,0.6278,0.0000
1,Ablated ModernBERT,0.6197,-0.0081


In [46]:
# top 500 features
shap_top_500 = shap_agg.sort_values("mean_abs_shap", ascending=False).head(500)
lime_top_500 = lime_global.head(500)

combined_500 = set(shap_top_500["token"]).union(set(lime_top_500["feature"]))
combined_500_df = pd.DataFrame({"feature": sorted(combined_500)})
combined_500_df["lexicon_overlap"] = combined_500_df["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)
ablation_terms_500 = combined_500_df[combined_500_df["lexicon_overlap"]]["feature"].tolist()

print("Extended ablation terms:", len(ablation_terms_500))

Extended ablation terms: 236


In [47]:
test_df_ablated_500 = test_df.copy()
test_df_ablated_500["post_cleaned"] = test_df_ablated_500["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_500)
)

preds_ablated_500 = []
texts_ablated_500 = test_df_ablated_500["post_cleaned"].tolist()

for i in range(0, len(texts_ablated_500), batch_size):
    batch = texts_ablated_500[i:i+batch_size]
    inputs = tokenizer(batch, truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_lora_final(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    preds_ablated_500.extend([id2label[p] for p in probs.argmax(axis=1)])

_, _, ablated_500_f1, _ = precision_recall_fscore_support(y_true, preds_ablated_500, average="macro", zero_division=0)

print("Original F1:", original_f1)
print("Ablated 500 F1:", ablated_500_f1)
print("Change:", round(ablated_500_f1 - original_f1, 4))

Original F1: 0.6278085808580858
Ablated 500 F1: 0.6223925539978894
Change: -0.0054


In [48]:
shap_top_1000 = shap_agg.sort_values("mean_abs_shap", ascending=False).head(1000)
lime_top_1000 = lime_global.head(1000)

combined_1000 = set(shap_top_1000["token"]).union(set(lime_top_1000["feature"]))
combined_1000_df = pd.DataFrame({"feature": sorted(combined_1000)})
combined_1000_df["lexicon_overlap"] = combined_1000_df["feature"].apply(
    lambda x: feature_overlaps_lexicon(x, political_lexicon)
)
ablation_terms_1000 = combined_1000_df[combined_1000_df["lexicon_overlap"]]["feature"].tolist()

print("Terms:", len(ablation_terms_1000))

Terms: 384


In [49]:
test_df_ablated_1000 = test_df.copy()
test_df_ablated_1000["post_cleaned"] = test_df_ablated_1000["post_cleaned"].apply(
    lambda x: remove_terms(x, ablation_terms_1000)
)

preds_ablated_1000 = []
texts_ablated_1000 = test_df_ablated_1000["post_cleaned"].tolist()

for i in range(0, len(texts_ablated_1000), batch_size):
    batch = texts_ablated_1000[i:i+batch_size]
    inputs = tokenizer(batch, truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_lora_final(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    preds_ablated_1000.extend([id2label[p] for p in probs.argmax(axis=1)])

_, _, ablated_1000_f1, _ = precision_recall_fscore_support(y_true, preds_ablated_1000, average="macro", zero_division=0)

print("Original F1:", round(original_f1, 4))
print("Ablated 1000 F1:", round(ablated_1000_f1, 4))
print("Change:", round(ablated_1000_f1 - original_f1, 4))

Original F1: 0.6278
Ablated 1000 F1: 0.6258
Change: -0.002


In [50]:
print(ablation_terms_1000[:50])

['abusive', 'account', 'activity', 'actual', 'additional', 'adult', 'afternoon', 'ahead', 'albert', 'alert', 'alex', 'algorithm', 'alleviate', 'american', 'ancient', 'anti', 'arabia', 'argument', 'army', 'arrive', 'aside', 'assange', 'asset', 'attorney', 'aware', 'awareness', 'balkan', 'bank', 'bed', 'belgian', 'beloved', 'big', 'bind', 'birthday', 'bizarre', 'bloc', 'blockade', 'board', 'bomb', 'boom', 'bosnia', 'breed', 'broadcast', 'business', 'camera', 'cannon', 'capitalist', 'cease', 'censor', 'censorship']


In [51]:
ablation_final = pd.DataFrame({
    "experiment": ["Original", "Ablation 127 terms (top 200)", "Ablation 236 terms (top 500)", "Ablation 384 terms (top 1000)"],
    "macro_f1": [round(original_f1, 4), round(ablated_f1, 4), round(ablated_500_f1, 4), round(ablated_1000_f1, 4)],
    "f1_change": [0, round(ablated_f1 - original_f1, 4), round(ablated_500_f1 - original_f1, 4), round(ablated_1000_f1 - original_f1, 4)]
})

ablation_final.to_csv("/content/drive/MyDrive/modernbert_ablation_results.csv", index=False)
ablation_final

,experiment,macro_f1,f1_change
0,Original,0.6278,0.0000
1,Ablation 127 terms (top 200),0.6197,-0.0081
2,Ablation 236 terms (top 500),0.6224,-0.0054
3,Ablation 384 terms (top 1000),0.6258,-0.0020
